# **test phase**

In [1]:
"""
SemEval Polarization Detection - Bengali Text
TEST Phase: Train on train.csv, predict on test.csv (unlabeled), create submission
"""

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_PATH = '/kaggle/input/semeval-polarization/test_phase_pola/subtask1/train/ben.csv'
TEST_PATH  = '/kaggle/input/semeval-polarization/test_phase_pola/subtask1/test/ben.csv'

# # Model choice - pick ONE:
# MODEL_NAME = 'xlm-roberta-base'  # Fast, multilingual, good baseline
# # MODEL_NAME = 'sagorsarker/bangla-bert-base'
# MODEL_NAME = "csebuetnlp/banglabert"# Bengali-specific, may be better
# MODEL_NAME = 'bert-base-multilingual-cased'  # Alternative
MODEL_NAME = 'google/muril-base-cased'

# ============================================================================
# DATASET CLASS
# ============================================================================

class PolarizationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)

    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'f1_class_0': f1_score(labels, preds, pos_label=0),
        'f1_class_1': f1_score(labels, preds, pos_label=1),
    }

# ============================================================================
# WEIGHTED TRAINER
# ============================================================================

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = None
        if self.class_weights is not None:
            weights = torch.tensor(self.class_weights, device=logits.device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("=" * 70)
    print("SemEval Polarization Detection - TEST Phase Pipeline")
    print("=" * 70)

    # ------------------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------------------
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)

    print(f"Train size: {len(train_df)}")
    print(f"Test size: {len(test_df)}")
    print("\nClass distribution:")
    print(train_df['polarization'].value_counts(normalize=True) * 100)

    # ------------------------------------------------------------------------
    # Internal validation split
    # ------------------------------------------------------------------------
    train_split, val_split = train_test_split(
        train_df,
        test_size=0.15,
        stratify=train_df['polarization'],
        random_state=42
    )

    # ------------------------------------------------------------------------
    # Model & tokenizer
    # ------------------------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    # ------------------------------------------------------------------------
    # Datasets
    # ------------------------------------------------------------------------
    train_dataset = PolarizationDataset(
        train_split['text'].values,
        train_split['polarization'].values,
        tokenizer
    )

    val_dataset = PolarizationDataset(
        val_split['text'].values,
        val_split['polarization'].values,
        tokenizer
    )

    # ------------------------------------------------------------------------
    # Class weights
    # ------------------------------------------------------------------------
    counts = train_split['polarization'].value_counts().sort_index()
    total = len(train_split)
    class_weights = [total / (2 * c) for c in counts]

    # ------------------------------------------------------------------------
    # Training (with validation)
    # ------------------------------------------------------------------------
    training_args = TrainingArguments(
        output_dir='./model_output',
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(3)],
        class_weights=class_weights
    )

    trainer.train()

    print("\nInternal validation results:")
    eval_res = trainer.evaluate()
    print(eval_res)

    # ------------------------------------------------------------------------
    # Retrain on FULL data
    # ------------------------------------------------------------------------
    print("\nRetraining on full training data...")

    model_final = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    full_dataset = PolarizationDataset(
        train_df['text'].values,
        train_df['polarization'].values,
        tokenizer
    )

    counts_full = train_df['polarization'].value_counts().sort_index()
    total_full = len(train_df)
    class_weights_full = [total_full / (2 * c) for c in counts_full]

    final_args = TrainingArguments(
        output_dir='./final_model',
        num_train_epochs=4,
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer_final = WeightedTrainer(
        model=model_final,
        args=final_args,
        train_dataset=full_dataset,
        class_weights=class_weights_full
    )

    trainer_final.train()
    trainer_final.save_model('./final_model/best')

    # ------------------------------------------------------------------------
    # Predict on TEST set
    # ------------------------------------------------------------------------
    print("\nGenerating test predictions...")

    test_dataset = PolarizationDataset(
        test_df['text'].values,
        np.zeros(len(test_df)),
        tokenizer
    )

    loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model_final.to(device)
    model_final.eval()

    preds, probs = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)

            outputs = model_final(input_ids=input_ids, attention_mask=attn)
            p = torch.softmax(outputs.logits, dim=1)

            preds.extend(torch.argmax(p, dim=1).cpu().numpy())
            probs.extend(p.cpu().numpy())

    # ------------------------------------------------------------------------
    # Submission
    # ------------------------------------------------------------------------
    submission = pd.DataFrame({
        'id': test_df['id'],
        'polarization': preds
    })

    submission.to_csv('muril_test.csv', index=False)

    analysis = test_df.copy()
    analysis['predicted_polarization'] = preds
    analysis['prob_0'] = [p[0] for p in probs]
    analysis['prob_1'] = [p[1] for p in probs]
    analysis['confidence'] = [max(p) for p in probs]

    analysis.to_csv('test_predictions_detailed.csv', index=False)

    print("\nFiles created:")
    print(" - submission.csv")
    print(" - test_predictions_detailed.csv")
    print(" - ./final_model/best")

    return submission

# ============================================================================
# RUN
# ============================================================================

if __name__ == "__main__":
    submission = main()


2026-01-29 12:10:24.933863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769688625.093888      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769688625.141142      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769688625.519383      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769688625.519423      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769688625.519426      24 computation_placer.cc:177] computation placer alr

SemEval Polarization Detection - TEST Phase Pipeline
Train size: 3333
Test size: 1501

Class distribution:
polarization
0    57.275728
1    42.724272
Name: proportion, dtype: float64


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


Internal validation results:


{'eval_loss': 0.5015018582344055, 'eval_macro_f1': 0.7928281622422146, 'eval_f1_class_0': 0.8255613126079447, 'eval_f1_class_1': 0.7600950118764845, 'eval_runtime': 4.6311, 'eval_samples_per_second': 107.966, 'eval_steps_per_second': 1.727, 'epoch': 4.0}

Retraining on full training data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss



Generating test predictions...

Files created:
 - submission.csv
 - test_predictions_detailed.csv
 - ./final_model/best


#  **Ensemble**

In [2]:
# import pandas as pd

# # =====================================================
# # CONFIG: paths to your submission files
# # =====================================================
# SUBMISSION_FILES = [
#     "/kaggle/input/semeval-polarization/pred_ben_transformers/pred_ben_banglabert.csv",
#     "/kaggle/input/semeval-polarization/pred_ben_transformers/pred_ben_multi.csv",
#     "/kaggle/input/semeval-polarization/pred_ben_transformers/pred_ben_muril.csv",
#     "/kaggle/input/semeval-polarization/pred_ben_transformers/pred_ben_xlm.csv",
#     "/kaggle/input/semeval-polarization/pred_ben_transformers/pred_ben_llama.csv",
# ]

# OUTPUT_PATH = "submission_ensemble.csv"

# # =====================================================
# # LOAD SUBMISSIONS
# # =====================================================
# subs = []
# for i, path in enumerate(SUBMISSION_FILES, start=1):
#     df = pd.read_csv(path)
#     df = df[['id', 'polarization']].rename(
#         columns={'polarization': f'm{i}'}
#     )
#     subs.append(df)

# # =====================================================
# # MERGE ALL ON id
# # =====================================================
# ensemble_df = subs[0]
# for df in subs[1:]:
#     ensemble_df = ensemble_df.merge(df, on='id', how='inner')

# # =====================================================
# # MAJORITY VOTE (tie → 0 by default)
# # =====================================================
# ensemble_df['polarization'] = (
#     ensemble_df[['m1', 'm2', 'm3', 'm4', 'm5']]
#     .mode(axis=1)[0]
#     .astype(int)
# )

# # =====================================================
# # FINAL SUBMISSION
# # =====================================================
# final_submission = ensemble_df[['id', 'polarization']]
# final_submission.to_csv(OUTPUT_PATH, index=False)

# # =====================================================
# # STATS
# # =====================================================
# print("Ensemble submission created:", OUTPUT_PATH)
# print("Total samples:", len(final_submission))
# print("Polarized (1):", (final_submission['polarization'] == 1).sum())
# print("Non-Polarized (0):", (final_submission['polarization'] == 0).sum())

# # =====================================================
# # OPTIONAL: AGREEMENT ANALYSIS
# # =====================================================
# ensemble_df['agreement'] = ensemble_df[['m1','m2','m3','m4','m5']].nunique(axis=1)
# print("\nAgreement breakdown:")
# print(ensemble_df['agreement'].value_counts().sort_index())


# **dev phase**

In [3]:
"""
# SemEval Polarization Detection - Bengali Text
# Dev Phase: Train on train.csv, predict on dev.csv (unlabeled), create submission
# """

# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import (
#     AutoTokenizer, AutoModelForSequenceClassification,
#     Trainer, TrainingArguments, EarlyStoppingCallback
# )
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import f1_score, classification_report
# import warnings
# warnings.filterwarnings('ignore')

# # ============================================================================
# # CONFIGURATION
# # ============================================================================

# # Paths (modify if needed)
# TRAIN_PATH = '/kaggle/input/semeval-polarization/subtask1/train/ben.csv'
# DEV_PATH = '/kaggle/input/semeval-polarization/subtask1/dev/ben.csv'

# # Model choice - pick ONE:
# # MODEL_NAME = 'xlm-roberta-base'  # Fast, multilingual, good baseline
# # MODEL_NAME = 'sagorsarker/bangla-bert-base'  # Bengali-specific, may be better
# # MODEL_NAME = 'bert-base-multilingual-cased'  # Alternative
# MODEL_NAME = 'google/muril-base-cased'


# # ============================================================================
# # DATASET CLASS
# # ============================================================================

# class PolarizationDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_length=256):
#         self.texts = texts
#         self.labels = labels
#         self.tokenizer = tokenizer
#         self.max_length = max_length
    
#     def __len__(self):
#         return len(self.texts)
    
#     def __getitem__(self, idx):
#         text = str(self.texts[idx])
#         label = self.labels[idx]
        
#         encoding = self.tokenizer(
#             text,
#             add_special_tokens=True,
#             max_length=self.max_length,
#             padding='max_length',
#             truncation=True,
#             return_attention_mask=True,
#             return_tensors='pt'
#         )
        
#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'labels': torch.tensor(label, dtype=torch.long)
#         }

# # ============================================================================
# # CUSTOM METRICS
# # ============================================================================

# def compute_metrics(eval_pred):
#     predictions, labels = eval_pred
#     predictions = np.argmax(predictions, axis=1)
    
#     macro_f1 = f1_score(labels, predictions, average='macro')
    
#     return {
#         'macro_f1': macro_f1,
#         'f1_class_0': f1_score(labels, predictions, pos_label=0),
#         'f1_class_1': f1_score(labels, predictions, pos_label=1)
#     }

# # ============================================================================
# # WEIGHTED TRAINER (for class imbalance)
# # ============================================================================

# class WeightedTrainer(Trainer):
#     def __init__(self, *args, class_weights=None, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.class_weights = class_weights
    
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.get("logits")
        
#         if self.class_weights is not None:
#             weights = torch.tensor(self.class_weights, device=logits.device)
#             loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
#         else:
#             loss_fct = torch.nn.CrossEntropyLoss()
        
#         loss = loss_fct(logits, labels)
#         return (loss, outputs) if return_outputs else loss

# # ============================================================================
# # MAIN TRAINING AND PREDICTION PIPELINE
# # ============================================================================

# def main():
#     print("="*70)
#     print("SemEval Polarization Detection - Dev Phase Submission Pipeline")
#     print("="*70)
    
#     # ========================================================================
#     # STEP 1: Load Data
#     # ========================================================================
#     print("\n[STEP 1] Loading data...")
#     train_df = pd.read_csv(TRAIN_PATH)
#     dev_df = pd.read_csv(DEV_PATH)
    
#     print(f"Training set size: {len(train_df)}")
#     print(f"Dev set size (unlabeled): {len(dev_df)}")
#     print(f"\nTraining class distribution:")
#     print(train_df['polarization'].value_counts())
#     print(f"\nClass percentages:")
#     print(train_df['polarization'].value_counts(normalize=True) * 100)
    
#     # ========================================================================
#     # STEP 2: Create Internal Validation Split (for monitoring during training)
#     # ========================================================================
#     print("\n[STEP 2] Creating internal validation split from training data...")
#     train_split, val_split = train_test_split(
#         train_df, 
#         test_size=0.15, 
#         stratify=train_df['polarization'], 
#         random_state=42
#     )
    
#     print(f"Training samples: {len(train_split)}")
#     print(f"Internal validation samples: {len(val_split)}")
    
#     # ========================================================================
#     # STEP 3: Load Tokenizer and Model
#     # ========================================================================
#     print(f"\n[STEP 3] Loading model: {MODEL_NAME}")
#     print("Estimated training time: ~15-20 minutes on GPU")
    
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     model = AutoModelForSequenceClassification.from_pretrained(
#         MODEL_NAME,
#         num_labels=2,
#         problem_type="single_label_classification"
#     )
    
#     # ========================================================================
#     # STEP 4: Create Datasets
#     # ========================================================================
#     print("\n[STEP 4] Creating datasets...")
    
#     train_dataset = PolarizationDataset(
#         train_split['text'].values,
#         train_split['polarization'].values,
#         tokenizer
#     )
    
#     val_dataset = PolarizationDataset(
#         val_split['text'].values,
#         val_split['polarization'].values,
#         tokenizer
#     )
    
#     # ========================================================================
#     # STEP 5: Calculate Class Weights
#     # ========================================================================
#     print("\n[STEP 5] Calculating class weights for imbalanced data...")
    
#     class_counts = train_split['polarization'].value_counts().sort_index()
#     total = len(train_split)
#     class_weights = [total / (2 * count) for count in class_counts]
#     print(f"Class weights: {class_weights}")
    
#     # ========================================================================
#     # STEP 6: Training
#     # ========================================================================
#     print("\n[STEP 6] Training model...")
    
#     training_args = TrainingArguments(
#         output_dir='./model_output',
#         num_train_epochs=4,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         warmup_steps=100,
#         learning_rate=2e-5,
#         weight_decay=0.01,
#         logging_dir='./logs',
#         logging_steps=50,
#         eval_strategy="steps",
#         eval_steps=100,
#         save_strategy="steps",
#         save_steps=100,
#         load_best_model_at_end=True,
#         metric_for_best_model='macro_f1',
#         greater_is_better=True,
#         save_total_limit=2,
#         fp16=torch.cuda.is_available(),
#         report_to="none"
#     )
    
#     trainer = WeightedTrainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#         compute_metrics=compute_metrics,
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
#         class_weights=class_weights
#     )
    
#     # Train
#     trainer.train()
    
#     # Evaluate on internal validation
#     print("\n[STEP 6 - Evaluation] Internal validation results:")
#     eval_results = trainer.evaluate()
#     print(f"Macro F1: {eval_results['eval_macro_f1']:.4f}")
#     print(f"F1 Class 0 (Non-Polarized): {eval_results['eval_f1_class_0']:.4f}")
#     print(f"F1 Class 1 (Polarized): {eval_results['eval_f1_class_1']:.4f}")
    
#     # Get predictions on internal validation
#     val_predictions = trainer.predict(val_dataset)
#     val_pred_labels = np.argmax(val_predictions.predictions, axis=1)
    
#     print("\nDetailed Classification Report (Internal Validation):")
#     print(classification_report(
#         val_split['polarization'].values,
#         val_pred_labels,
#         target_names=['Non-Polarized', 'Polarized'],
#         digits=4
#     ))
    
#     # ========================================================================
#     # STEP 7: Retrain on FULL training data for final submission
#     # ========================================================================
#     print("\n[STEP 7] Retraining on FULL training data for final predictions...")
#     print("This ensures we use all available training data.")
    
#     # Reload fresh model
#     model_final = AutoModelForSequenceClassification.from_pretrained(
#         MODEL_NAME,
#         num_labels=2,
#         problem_type="single_label_classification"
#     )
    
#     # Create dataset with FULL training data
#     full_train_dataset = PolarizationDataset(
#         train_df['text'].values,
#         train_df['polarization'].values,
#         tokenizer
#     )
    
#     # Recalculate class weights on full data
#     class_counts_full = train_df['polarization'].value_counts().sort_index()
#     total_full = len(train_df)
#     class_weights_full = [total_full / (2 * count) for count in class_counts_full]
    
#     training_args_final = TrainingArguments(
#         output_dir='./final_model',
#         num_train_epochs=4,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         warmup_steps=100,
#         learning_rate=2e-5,
#         weight_decay=0.01,
#         logging_steps=50,
#         save_strategy="epoch",
#         save_total_limit=1,
#         fp16=torch.cuda.is_available(),
#         report_to="none"
#     )
    
#     trainer_final = WeightedTrainer(
#         model=model_final,
#         args=training_args_final,
#         train_dataset=full_train_dataset,
#         class_weights=class_weights_full
#     )
    
#     trainer_final.train()
    
#     # Save final model
#     trainer_final.save_model('./final_model/best')
#     print("Final model saved to './final_model/best'")
    
#     # ========================================================================
#     # STEP 8: Predict on Dev Set (Unlabeled)
#     # ========================================================================
#     print("\n[STEP 8] Generating predictions on dev set...")
    
#     # Create dataset for dev (with dummy labels since we don't have real labels)
#     dev_dataset = PolarizationDataset(
#         dev_df['text'].values,
#         np.zeros(len(dev_df)),  # Dummy labels
#         tokenizer
#     )
    
#     # Get predictions
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     model_final.to(device)
#     model_final.eval()
    
#     dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
    
#     all_predictions = []
#     all_probabilities = []
    
#     print("Predicting...")
#     with torch.no_grad():
#         for batch in dev_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
            
#             outputs = model_final(input_ids=input_ids, attention_mask=attention_mask)
#             probs = torch.softmax(outputs.logits, dim=1)
#             preds = torch.argmax(outputs.logits, dim=1)
            
#             all_predictions.extend(preds.cpu().numpy())
#             all_probabilities.extend(probs.cpu().numpy())
    
#     # ========================================================================
#     # STEP 9: Create Submission File
#     # ========================================================================
#     print("\n[STEP 9] Creating submission file...")
    
#     submission = pd.DataFrame({
#         'id': dev_df['id'],
#         'polarization': all_predictions
#     })
    
#     submission.to_csv('submission.csv', index=False)
    
#     print("\n" + "="*70)
#     print("SUBMISSION FILE CREATED: submission.csv")
#     print("="*70)
#     print(f"\nTotal predictions: {len(submission)}")
#     print(f"Predicted as Polarized (1): {(submission['polarization'] == 1).sum()} ({(submission['polarization'] == 1).sum() / len(submission) * 100:.2f}%)")
#     print(f"Predicted as Non-Polarized (0): {(submission['polarization'] == 0).sum()} ({(submission['polarization'] == 0).sum() / len(submission) * 100:.2f}%)")
    
#     print("\nSubmission format preview:")
#     print(submission.head(10))
    
#     # Save detailed analysis file with probabilities
#     dev_analysis = dev_df.copy()
#     dev_analysis['predicted_polarization'] = all_predictions
#     dev_analysis['probability_class_0'] = [p[0] for p in all_probabilities]
#     dev_analysis['probability_class_1'] = [p[1] for p in all_probabilities]
#     dev_analysis['confidence'] = [max(p) for p in all_probabilities]
#     dev_analysis.to_csv('dev_predictions_detailed.csv', index=False)
    
#     print("\nDetailed predictions saved to: dev_predictions_detailed.csv")
#     print("(Includes probabilities and confidence scores)")
    
#     # ========================================================================
#     # Summary
#     # ========================================================================
#     print("\n" + "="*70)
#     print("PIPELINE COMPLETE!")
#     print("="*70)
#     print("\nFiles created:")
#     print("  1. submission.csv - Submit this file to the competition")
#     print("  2. dev_predictions_detailed.csv - For your analysis")
#     print("  3. ./final_model/best/ - Trained model (for test phase)")
    
#     print(f"\nInternal validation Macro F1: {eval_results['eval_macro_f1']:.4f}")
#     print("(This is an estimate of your expected competition score)")
    
#     print("\nNext steps:")
#     print("  - Submit 'submission.csv' to the competition")
#     print("  - Check leaderboard score")
#     print("  - If score < 80%, try different model or ensemble")
#     print("  - Analyze dev_predictions_detailed.csv for insights")
    
#     return submission

# # ============================================================================
# # RUN THE PIPELINE
# # ============================================================================

# if __name__ == "__main__":
#     submission = main()

'\n# SemEval Polarization Detection - Bengali Text\n# Dev Phase: Train on train.csv, predict on dev.csv (unlabeled), create submission\n# '

In [4]:
# """
# SemEval Polarization Detection - Bengali Text
# Complete solution with multiple transformer approaches
# """

# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import (
#     AutoTokenizer, AutoModelForSequenceClassification,
#     Trainer, TrainingArguments, EarlyStoppingCallback
# )
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import f1_score, classification_report
# import warnings
# warnings.filterwarnings('ignore')

# # ============================================================================
# # DATASET CLASS
# # ============================================================================

# class PolarizationDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_length=256):
#         self.texts = texts
#         self.labels = labels
#         self.tokenizer = tokenizer
#         self.max_length = max_length
    
#     def __len__(self):
#         return len(self.texts)
    
#     def __getitem__(self, idx):
#         text = str(self.texts[idx])
#         label = self.labels[idx]
        
#         encoding = self.tokenizer(
#             text,
#             add_special_tokens=True,
#             max_length=self.max_length,
#             padding='max_length',
#             truncation=True,
#             return_attention_mask=True,
#             return_tensors='pt'
#         )
        
#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'labels': torch.tensor(label, dtype=torch.long)
#         }

# # ============================================================================
# # CUSTOM METRICS
# # ============================================================================

# def compute_metrics(eval_pred):
#     predictions, labels = eval_pred
#     predictions = np.argmax(predictions, axis=1)
    
#     macro_f1 = f1_score(labels, predictions, average='macro')
    
#     return {
#         'macro_f1': macro_f1,
#         'f1_class_0': f1_score(labels, predictions, pos_label=0),
#         'f1_class_1': f1_score(labels, predictions, pos_label=1)
#     }

# # ============================================================================
# # WEIGHTED TRAINER (for class imbalance)
# # ============================================================================

# class WeightedTrainer(Trainer):
#     def __init__(self, *args, class_weights=None, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.class_weights = class_weights
    
#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.get("logits")
        
#         if self.class_weights is not None:
#             weights = torch.tensor(self.class_weights, device=logits.device)
#             loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
#         else:
#             loss_fct = torch.nn.CrossEntropyLoss()
        
#         loss = loss_fct(logits, labels)
#         return (loss, outputs) if return_outputs else loss

# # ============================================================================
# # TRAINING FUNCTION
# # ============================================================================

# def train_model(train_df, val_df, model_name, output_dir, use_weights=True):
#     """
#     Train a single model
    
#     Recommended models for Bengali:
#     - 'xlm-roberta-base' (multilingual, good baseline)
#     - 'bert-base-multilingual-cased' (multilingual BERT)
#     - 'ai4bharat/indic-bert' (Indic languages including Bengali)
#     - 'sagorsarker/bangla-bert-base' (Bengali-specific)
#     """
    
#     # Load tokenizer and model
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=2,
#         problem_type="single_label_classification"
#     )
    
#     # Create datasets
#     train_dataset = PolarizationDataset(
#         train_df['text'].values,
#         train_df['polarization'].values,
#         tokenizer
#     )
    
#     val_dataset = PolarizationDataset(
#         val_df['text'].values,
#         val_df['polarization'].values,
#         tokenizer
#     )
    
#     # Calculate class weights
#     class_weights = None
#     if use_weights:
#         class_counts = train_df['polarization'].value_counts().sort_index()
#         total = len(train_df)
#         class_weights = [total / (2 * count) for count in class_counts]
#         print(f"Class weights: {class_weights}")
    
#     # Training arguments
#     training_args = TrainingArguments(
#         output_dir=output_dir,
#         num_train_epochs=5,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         warmup_steps=100,
#         learning_rate=2e-5,
#         weight_decay=0.01,
#         logging_dir=f'{output_dir}/logs',
#         logging_steps=50,
#         eval_strategy="steps",
#         eval_steps=100,
#         save_strategy="steps",
#         save_steps=100,
#         load_best_model_at_end=True,
#         metric_for_best_model='macro_f1',
#         greater_is_better=True,
#         save_total_limit=2,
#         fp16=torch.cuda.is_available(),
#         report_to="none"
#     )
    
#     # Initialize trainer
#     trainer_class = WeightedTrainer if use_weights else Trainer
#     trainer = trainer_class(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#         compute_metrics=compute_metrics,
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
#         class_weights=class_weights if use_weights else None
#     )
    
#     # Train
#     print(f"\nTraining {model_name}...")
#     trainer.train()
    
#     # Evaluate
#     eval_results = trainer.evaluate()
#     print(f"\nValidation Results:")
#     print(f"Macro F1: {eval_results['eval_macro_f1']:.4f}")
#     print(f"F1 Class 0: {eval_results['eval_f1_class_0']:.4f}")
#     print(f"F1 Class 1: {eval_results['eval_f1_class_1']:.4f}")
    
#     return trainer, eval_results

# # ============================================================================
# # K-FOLD CROSS VALIDATION
# # ============================================================================

# def train_kfold(df, model_name, n_splits=5):
#     """
#     K-Fold Cross Validation for robust evaluation
#     """
#     skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
#     fold_scores = []
    
#     for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['polarization'])):
#         print(f"\n{'='*60}")
#         print(f"FOLD {fold + 1}/{n_splits}")
#         print(f"{'='*60}")
        
#         train_df = df.iloc[train_idx].reset_index(drop=True)
#         val_df = df.iloc[val_idx].reset_index(drop=True)
        
#         trainer, eval_results = train_model(
#             train_df, val_df,
#             model_name,
#             output_dir=f'./output/fold_{fold}',
#             use_weights=True
#         )
        
#         fold_scores.append(eval_results['eval_macro_f1'])
    
#     print(f"\n{'='*60}")
#     print(f"CROSS VALIDATION RESULTS")
#     print(f"{'='*60}")
#     print(f"Macro F1 scores: {fold_scores}")
#     print(f"Mean: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
    
#     return fold_scores

# # ============================================================================
# # ENSEMBLE PREDICTION
# # ============================================================================

# def ensemble_predict(models_and_tokenizers, texts, device='cuda'):
#     """
#     Ensemble prediction from multiple models
#     """
#     all_predictions = []
    
#     for model, tokenizer in models_and_tokenizers:
#         model.to(device)
#         model.eval()
        
#         predictions = []
        
#         with torch.no_grad():
#             for text in texts:
#                 encoding = tokenizer(
#                     text,
#                     add_special_tokens=True,
#                     max_length=256,
#                     padding='max_length',
#                     truncation=True,
#                     return_tensors='pt'
#                 ).to(device)
                
#                 outputs = model(**encoding)
#                 probs = torch.softmax(outputs.logits, dim=1)
#                 predictions.append(probs.cpu().numpy())
        
#         all_predictions.append(np.vstack(predictions))
    
#     # Average probabilities
#     avg_probs = np.mean(all_predictions, axis=0)
#     final_preds = np.argmax(avg_probs, axis=1)
    
#     return final_preds

# # ============================================================================
# # MAIN EXECUTION
# # ============================================================================

# if __name__ == "__main__":
#     # Load data
#     print("Loading data...")
#     df = pd.read_csv('train.csv')
    
#     print(f"Dataset size: {len(df)}")
#     print(f"Class distribution:\n{df['polarization'].value_counts()}")
#     print(f"Class distribution (%):\n{df['polarization'].value_counts(normalize=True) * 100}")
    
#     # ========================================================================
#     # APPROACH 1: Single Best Model (Quick Start)
#     # ========================================================================
#     print("\n" + "="*60)
#     print("APPROACH 1: Single Model Training")
#     print("="*60)
    
#     # Split data
#     from sklearn.model_selection import train_test_split
#     train_df, val_df = train_test_split(
#         df, test_size=0.15, stratify=df['polarization'], random_state=42
#     )
    
#     # Try the best model for Bengali
#     # Recommended: 'xlm-roberta-base' or 'sagorsarker/bangla-bert-base'
#     best_model = 'xlm-roberta-base'
    
#     trainer, results = train_model(
#         train_df, val_df,
#         model_name=best_model,
#         output_dir='./best_model',
#         use_weights=True
#     )
    
#     # Save the best model
#     trainer.save_model('./best_model/final')
    
#     # ========================================================================
#     # APPROACH 2: K-Fold Cross Validation (More Robust)
#     # ========================================================================
#     print("\n" + "="*60)
#     print("APPROACH 2: K-Fold Cross Validation")
#     print("="*60)
    
#     # Uncomment to run k-fold
#     # fold_scores = train_kfold(df, model_name='xlm-roberta-base', n_splits=5)
    
#     # ========================================================================
#     # APPROACH 3: Multi-Model Ensemble (Highest Performance)
#     # ========================================================================
#     print("\n" + "="*60)
#     print("APPROACH 3: Ensemble Training")
#     print("="*60)
    
#     # Train multiple models
#     models_to_ensemble = [
#         'xlm-roberta-base',
#         'bert-base-multilingual-cased',
#         # 'sagorsarker/bangla-bert-base',  # Uncomment if you want Bengali-specific
#     ]
    
#     # Uncomment to train ensemble
#     # for i, model_name in enumerate(models_to_ensemble):
#     #     print(f"\nTraining ensemble model {i+1}/{len(models_to_ensemble)}: {model_name}")
#     #     train_model(train_df, val_df, model_name, f'./ensemble/model_{i}')
    
#     print("\n" + "="*60)
#     print("Training Complete!")
#     print("="*60)
#     print("\nFor inference on test data, use:")
#     print("  - Load saved model from './best_model/final'")
#     print("  - Tokenize test texts")
#     print("  - Get predictions and save as submission.csv")

# # ============================================================================
# # INFERENCE CODE (for test data)
# # ============================================================================

# def predict_test_data(test_csv_path, model_path, output_path='submission.csv'):
#     """
#     Generate predictions for test data
#     """
#     # Load test data
#     test_df = pd.read_csv(test_csv_path)
    
#     # Load model and tokenizer
#     tokenizer = AutoTokenizer.from_pretrained(model_path)
#     model = AutoModelForSequenceClassification.from_pretrained(model_path)
    
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     model.to(device)
#     model.eval()
    
#     # Create dataset
#     test_dataset = PolarizationDataset(
#         test_df['text'].values,
#         np.zeros(len(test_df)),  # Dummy labels
#         tokenizer
#     )
    
#     # DataLoader
#     test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
#     # Predict
#     predictions = []
#     with torch.no_grad():
#         for batch in test_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
            
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#             preds = torch.argmax(outputs.logits, dim=1)
#             predictions.extend(preds.cpu().numpy())
    
#     # Create submission
#     submission = pd.DataFrame({
#         'id': test_df['id'],
#         'polarization': predictions
#     })
    
#     submission.to_csv(output_path, index=False)
#     print(f"Predictions saved to {output_path}")
    
#     return submission

# # Usage:
# # submission = predict_test_data('test.csv', './best_model/final', 'submission.csv')